# ASCIP — Data Exploration & First Forecasting Model

This notebook covers the first ML stage of ASCIP:

1. Load and inspect the supply-chain dataset
2. Clean and prepare dates
3. Perform basic EDA
4. Explore demand over time
5. Select one product-warehouse demand series
6. Split time-series data chronologically
7. Build a naive baseline
8. Create lag and rolling features
9. Train a Random Forest forecasting model
10. Evaluate the model
11. Visualize predictions
12. Detect demand anomalies with Isolation Forest
13. Perform basic supplier-risk analysis

The first forecasting example uses **Laptop (P001) in Delhi**. Later we will generalize the pipeline to all product-warehouse combinations.


## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")


## 2. Load the raw dataset

In [ ]:
DATA_PATH = Path("../data/raw/supply_chain_data.csv")

df = pd.read_csv(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])

df.head()


In [ ]:
print("Shape:", df.shape)
print("Rows:", len(df))
print("Columns:", len(df.columns))


## 3. Basic data inspection

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

In [ ]:
print("Products:", df["product_id"].nunique())
print("Warehouses:", df["warehouse"].nunique())
print("Suppliers:", df["supplier_id"].nunique())
print("Categories:", df["category"].nunique())

print("\nWarehouses:")
print(df["warehouse"].value_counts())

print("\nSuppliers:")
print(df["supplier_id"].value_counts())


## 4. Total demand by product

In [ ]:
product_demand = (
    df.groupby("product_name")["demand"]
    .sum()
    .sort_values(ascending=False)
)

product_demand


In [ ]:
plt.figure(figsize=(12, 5))
product_demand.plot(kind="bar")
plt.title("Total Demand by Product")
plt.xlabel("Product")
plt.ylabel("Total Demand")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 5. Total daily demand

In [ ]:
daily_demand = df.groupby("date")["demand"].sum()

plt.figure(figsize=(14, 5))
plt.plot(daily_demand.index, daily_demand.values)
plt.title("Total Daily Demand")
plt.xlabel("Date")
plt.ylabel("Demand")
plt.tight_layout()
plt.show()


## 6. Select one product-warehouse series

For the first forecasting model we use Laptop (P001) in Delhi. This gives us one daily time series.


In [ ]:
product_data = df[
    (df["product_id"] == "P001") &
    (df["warehouse"] == "Delhi")
].copy()

product_data = product_data.sort_values("date").reset_index(drop=True)

product_data[["date", "product_id", "product_name", "warehouse", "demand", "inventory"]].head()


In [ ]:
product_data["rolling_7"] = product_data["demand"].rolling(7).mean()

plt.figure(figsize=(14, 5))
plt.plot(product_data["date"], product_data["demand"], alpha=0.45, label="Daily demand")
plt.plot(product_data["date"], product_data["rolling_7"], label="7-day rolling average")
plt.title("Laptop Demand - Delhi")
plt.xlabel("Date")
plt.ylabel("Demand")
plt.legend()
plt.tight_layout()
plt.show()


## 7. Time-series train/test split

We preserve chronological order.

- First 80% → training
- Last 20% → testing

We do not randomly shuffle time-series observations.


In [ ]:
split = int(len(product_data) * 0.8)

train = product_data.iloc[:split].copy()
test = product_data.iloc[split:].copy()

print("Training rows:", len(train))
print("Testing rows:", len(test))
print("Training period:", train["date"].min(), "to", train["date"].max())
print("Testing period:", test["date"].min(), "to", test["date"].max())


## 8. Baseline forecast

Baseline: tomorrow's demand is predicted as today's demand.

A more complicated model should beat this baseline.


In [ ]:
test["baseline_prediction"] = test["demand"].shift(1)
test.loc[test.index[0], "baseline_prediction"] = train["demand"].iloc[-1]

baseline_mae = mean_absolute_error(test["demand"], test["baseline_prediction"])
baseline_rmse = np.sqrt(mean_squared_error(test["demand"], test["baseline_prediction"]))

print("Baseline MAE:", round(baseline_mae, 2))
print("Baseline RMSE:", round(baseline_rmse, 2))


## 9. Feature engineering

Features use information available before the prediction date:

- lag 1 day
- lag 7 days
- 7-day rolling mean
- 14-day rolling mean
- day of week
- month
- promotion
- inventory

Rolling features are shifted by one day to avoid target leakage.


In [ ]:
forecast_df = product_data.copy()

forecast_df["lag_1"] = forecast_df["demand"].shift(1)
forecast_df["lag_7"] = forecast_df["demand"].shift(7)
forecast_df["rolling_7"] = forecast_df["demand"].shift(1).rolling(7).mean()
forecast_df["rolling_14"] = forecast_df["demand"].shift(1).rolling(14).mean()
forecast_df["day_of_week"] = forecast_df["date"].dt.dayofweek
forecast_df["month"] = forecast_df["date"].dt.month

forecast_df = forecast_df.dropna().reset_index(drop=True)

features = [
    "lag_1",
    "lag_7",
    "rolling_7",
    "rolling_14",
    "day_of_week",
    "month",
    "promotion",
    "inventory"
]

target = "demand"

forecast_df[features + [target]].head()


## 10. Chronological split for the ML model

In [ ]:
split = int(len(forecast_df) * 0.8)

train_ml = forecast_df.iloc[:split].copy()
test_ml = forecast_df.iloc[split:].copy()

X_train = train_ml[features]
y_train = train_ml[target]

X_test = test_ml[features]
y_test = test_ml[target]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training period:", train_ml["date"].min(), "to", train_ml["date"].max())
print("Testing period:", test_ml["date"].min(), "to", test_ml["date"].max())


## 11. Train the first ASCIP forecasting model

Random Forest is our first ML model because it can model nonlinear relationships and is easy to inspect.


In [ ]:
model = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
predictions = model.predict(X_test)


## 12. Evaluate the forecasting model

In [ ]:
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("Random Forest")
print("MAE :", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R²  :", round(r2, 4))

print("\nBaseline")
print("MAE :", round(baseline_mae, 2))
print("RMSE:", round(baseline_rmse, 2))


## 13. Actual vs predicted demand

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(test_ml["date"], y_test.values, label="Actual")
plt.plot(test_ml["date"], predictions, label="Predicted")
plt.title("Laptop Demand Forecast - Delhi")
plt.xlabel("Date")
plt.ylabel("Demand")
plt.legend()
plt.tight_layout()
plt.show()


## 14. Forecast errors

In [ ]:
results = test_ml[["date", "demand"]].copy()
results["prediction"] = predictions
results["error"] = results["demand"] - results["prediction"]
results["absolute_error"] = results["error"].abs()

results.head(10)


In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(results["date"], results["error"])
plt.axhline(0, linestyle="--")
plt.title("Forecast Error")
plt.xlabel("Date")
plt.ylabel("Actual - Predicted")
plt.tight_layout()
plt.show()


## 15. Feature importance

In [ ]:
importance = (
    pd.Series(model.feature_importances_, index=features)
    .sort_values(ascending=False)
)

print(importance)

plt.figure(figsize=(9, 5))
importance.sort_values().plot(kind="barh")
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()


## 16. Demand anomaly detection

Isolation Forest identifies observations that are unusual compared with the rest of the data.

For this first demonstration we use demand, inventory, delivery time and defect rate.


In [ ]:
anomaly_features = [
    "demand",
    "inventory",
    "delivery_days",
    "defect_rate"
]

anomaly_data = product_data[anomaly_features].copy()

anomaly_model = IsolationForest(
    contamination=0.02,
    random_state=42
)

anomaly_data["anomaly"] = anomaly_model.fit_predict(anomaly_data)
anomaly_data["anomaly"] = anomaly_data["anomaly"].map({1: 0, -1: 1})

product_data["anomaly"] = anomaly_data["anomaly"].values

print("Anomalies detected:", int(product_data["anomaly"].sum()))


In [ ]:
normal = product_data[product_data["anomaly"] == 0]
anomalies = product_data[product_data["anomaly"] == 1]

plt.figure(figsize=(14, 5))
plt.plot(normal["date"], normal["demand"], alpha=0.5, label="Normal")
plt.scatter(anomalies["date"], anomalies["demand"], label="Anomaly", zorder=3)
plt.title("Demand Anomalies - Laptop Delhi")
plt.xlabel("Date")
plt.ylabel("Demand")
plt.legend()
plt.tight_layout()
plt.show()


## 17. Supplier risk exploration

In [ ]:
supplier_summary = (
    df.groupby("supplier_id")
    .agg(
        avg_delivery_days=("delivery_days", "mean"),
        avg_defect_rate=("defect_rate", "mean"),
        on_time_rate=("supplier_on_time", "mean"),
        total_orders=("order_quantity", lambda x: (x > 0).sum()),
        total_stockouts=("stockout", "sum")
    )
    .sort_values("avg_delivery_days", ascending=False)
)

supplier_summary


## 18. Save the first forecasting model

In [ ]:
import joblib

MODEL_PATH = Path("../models/demand_forecaster.joblib")
joblib.dump(model, MODEL_PATH)

print("Saved:", MODEL_PATH)


## 19. Save processed forecasting data

In [ ]:
PROCESSED_PATH = Path("../data/processed/forecasting_data.csv")

forecast_df.to_csv(PROCESSED_PATH, index=False)

print("Saved:", PROCESSED_PATH)


# What we have achieved

The first ASCIP intelligence layer now contains:

- Data inspection
- Basic EDA
- Demand visualization
- Chronological train/test split
- Naive forecasting baseline
- Lag and rolling features
- Random Forest demand forecasting
- MAE/RMSE/R² evaluation
- Forecast visualization
- Feature importance
- Isolation Forest anomaly detection
- Supplier performance summary
- Saved forecasting model
- Saved processed forecasting data

## Next stage

We will generalize the forecasting pipeline across product-warehouse combinations, then build supplier-risk prediction and SHAP explanations.
